# Dealer Gamma 02 — build the dealer-gamma grid (cached)

Pre-registration: `discussions/Dealer_Gamma/001_claude_branch_preregistration.md`. Code: `src/data_engineering/modeling/dealer_gamma.py`.

One sequential pass per currency over every day of option and perpetual prints:

- net customer position per instrument (taker = customer), last traded iv, BS gamma on the Deribit index;
- 5-minute grid: dealer gamma $G$ (USD per 1% move) for the five variants, plus the part expiring within 24 h; perpetual last price and taker buy/sell USD;
- strike table at 05:00 UTC for the expiry of 08:00 the same day (pinning), and the 07:30–08:00 index average (settlement proxy).

The cache lives outside the repo (`DG_CACHE`, default `DERIBIT_DATA_ROOT\derived\dealer_gamma`). It is reused unless the archive has new days or `DG_FORCE=1`. Expect several minutes per currency.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import dealer_gamma as dg

ENV = dg.notebook_env('02', PROJECT_ROOT)
OUT, CACHE, HIST = ENV['out'], ENV['cache'], ENV['history']
PERIODS = ['discovery'] + (['holdout'] if ENV['holdout'] else [])
print(dg.VERSION, '| synthetic' if ENV['synthetic'] else '', '| periods', PERIODS)
print('data', ENV['data_root']); print('cache', CACHE); print('out', OUT)

In [ ]:
for c in dg.CURRENCIES:
    p = dg.cache_paths(CACHE, c)
    start, end = (dg.SYNTHETIC_DAYS if ENV['synthetic'] else dg.available_days(HIST, c))
    meta = json.loads(p['meta'].read_text()) if p['meta'].is_file() else {}
    if meta.get('end') == end and meta.get('version') == dg.VERSION and not ENV['force']:
        print(c, 'cache up to date', start, '->', end); continue
    T0 = time.time()
    grid, pins, settle, cov = dg.build(HIST, c, start, end, log=lambda m: print(' ', m, f'{time.time()-T0:.0f}s', flush=True))
    grid.to_parquet(p['grid'], index=False); pins.to_parquet(p['pins'], index=False)
    settle.to_parquet(p['settle'], index=False); cov.to_parquet(p['coverage'], index=False)
    dg.dump({'version': dg.VERSION, 'currency': c, 'start': start, 'end': end, 'rows': len(grid),
             'built_s': round(time.time() - T0)}, p['meta'])
    print(c, 'built', len(grid), 'rows', f'{time.time()-T0:.0f}s')

## Coverage and daily summary

In [ ]:
D = {c: dg.load_cache(CACHE, c) for c in dg.CURRENCIES}
cov = pd.concat([d['coverage'] for d in D.values()])
cov.to_csv(OUT / '02_coverage.csv', index=False)
print(cov.groupby('currency').agg(days=('day', 'size'), option_files=('option_file', 'sum'),
                                  perp_files=('perp_file', 'sum'), median_option_prints=('option_prints', 'median'),
                                  median_block_share=('option_block_share', 'median')))
print('missing option days:', cov[~cov.option_file][['currency', 'day']].values.tolist()[:20])
daily = []
for c, d in D.items():
    g = d['grid']
    x = g[(g.ts.dt.hour == dg.EXPIRY_HOUR) & (g.ts.dt.minute == 5)].copy()
    x['currency'] = c
    daily.append(x)
daily = pd.concat(daily)
daily.to_csv(OUT / '02_gex_daily.csv', index=False, float_format='%.6g')
corr = pd.concat({c: daily[daily.currency == c][[f'gex_{v}' for v in dg.VARIANTS]].corr() for c in dg.CURRENCIES})
corr.to_csv(OUT / '02_variant_corr.csv'); display(corr.round(2))
print('share of sessions with dealers long gamma (recon):', daily.groupby('currency').gex_recon.apply(lambda s: (s > 0).mean()).round(3).to_dict())

In [ ]:
fig, ax = plt.subplots(len(dg.CURRENCIES), 1, figsize=(14, 8), sharex=True)
for j, c in enumerate(dg.CURRENCIES):
    z = daily[daily.currency == c].set_index('ts')
    ax[j].plot(z.index, z.gex_recon / 1e6, lw=0.8, label='recon (primary)')
    ax[j].plot(z.index, z.gex_recon_noblock / 1e6, lw=0.6, alpha=0.7, label='recon_noblock')
    ax[j].axhline(0, color='k', lw=0.6); ax[j].set_ylabel('USD m per 1% move')
    a2 = ax[j].twinx(); a2.plot(z.index, z.S, color='grey', lw=0.6); a2.set_ylabel('index')
    for p0 in (dg.DISCOVERY[0], dg.HOLDOUT_START):
        ax[j].axvline(pd.Timestamp(p0, tz='UTC'), color='r', ls='--', lw=0.8)
    ax[j].set_title(f'{c}: dealer gamma at 08:05 UTC (grey: index)'); ax[j].legend(loc='upper left')
plt.tight_layout(); plt.savefig(OUT / '02_gex_daily.png', dpi=110); plt.show()